In [1]:
import requests

# v4 精准解析需要有效 Token（https://mineru.net/apiManage/token）。
# 当前 Token 已于 2026-07-13 过期，会返回 401 A0202。
# 这里改用官方免登录 Agent 轻量解析 API，流程仍然是：提交任务 → 拿到 task_id → 查询结果。
url = "https://mineru.net/api/v1/agent/parse/url"
data = {
    "url": "https://vl-image.oss-cn-shanghai.aliyuncs.com/Qwen3-tech_report.pdf",
    "is_ocr": True,
    "enable_formula": False,
}

res = requests.post(url, json=data)
print(res.status_code)
print(res.json())
print(res.json()["data"])

200
{'code': 0, 'msg': 'ok', 'trace_id': 'fa5e4c4f482bd67d19dafd366f5fa9ae', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406'}


In [2]:
payload = res.json()
if not payload.get("data"):
    raise RuntimeError(f"提交任务失败: {payload}")
task_id = payload["data"]["task_id"]
print(task_id) 

2ad787ff-203c-4984-84b8-85e314fe823406


In [3]:
## 获取任务结果（任务是异步的，需要轮询到 done / failed）
import time

url = f"https://mineru.net/api/v1/agent/parse/{task_id}"
data = {}
for _ in range(30):
    res = requests.get(url)
    body = res.json()
    print(res.status_code)
    print(body)
    data = body.get("data") or {}
    print(data)
    if data.get("state") in ("done", "failed"):
        break
    time.sleep(3)

if data.get("markdown_url"):
    md = requests.get(data["markdown_url"]).text
    print(md[:1500])

200
{'code': 0, 'msg': 'ok', 'trace_id': '29743bb8a95223e11d347805837d7ba6', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'pending'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'pending'}


200
{'code': 0, 'msg': 'ok', 'trace_id': 'e7678475b271526b7e90a7b5e42a9523', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'pending'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'pending'}


200
{'code': 0, 'msg': 'ok', 'trace_id': 'cfb0ec1bef8282ad29e3b364d2fe99b5', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': '6d0979dff647d5bedeeac40a3e88382d', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': 'b539cb51105f236da231354ef7739ae4', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': 'be885b3d3a6ee728a5e79d538ed7f954', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': '790a583ebc39f7e9c2924a8bbbb25439', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': '65f84bc958fadd26bffd33785d896391', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'running'}


200
{'code': 0, 'msg': 'ok', 'trace_id': 'ce939816744690d2d8d1183049031bc4', 'data': {'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'done', 'markdown_url': 'https://cdn-mineru.openxlab.org.cn/pdf/2026-08-29/b39d2eff-8fd6-407a-aa60-728ff7b3eaef/full.md'}}
{'task_id': '2ad787ff-203c-4984-84b8-85e314fe823406', 'state': 'done', 'markdown_url': 'https://cdn-mineru.openxlab.org.cn/pdf/2026-08-29/b39d2eff-8fd6-407a-aa60-728ff7b3eaef/full.md'}


<!-- image-->

# Qwen3:思深, 行速

2025年4月29日·4分钟·794字·Qwen Team|语言:English

<!-- image-->

QWEN CHAT ^ GITHUB > HUGGING FACE > MODELSCOPE ^ KAGGLE ^

DEMO ^ DISCORD ^

## 引言

今天，我们宣布推出Qwen3，这是Qwen 系列大型语言模型的最新成员。我们的旗舰模型Qwen3-235B-A22B 在代码、数学、通用能力等基准测试中，与 DeepSeek-R1、o1、o3-mini、Grok-3 和Gemini-2.5-Pro 等顶级模型相比，表现出极具竞争力的结果。此外，小型MoE模型Qwen3-30B-A3B的激活参数数量是QwQ-32B的10%，表现更胜一筹，甚至像Qwen3-4B 这样的小模型也能匹敌 Qwen2.5-72B-Instruct 的性能。

<table><tr><td colspan="5"><img src="images/b80df602cf3c36d3998d53aa22429715013eafd2b9ec89a508a24ebf5347fab8.jpg"/></td><td rowspan="2">Blog Publication Think</td><td rowspan="2">About</td><td rowspan="2">Try Qwen Chat Medium</td></tr><tr><td></td><td>MoE</td><td>Dense</td><td>2024-12-17</td><td></td></tr><tr><td>ArenaHard</td><td>95.6</td><td>93.8</td><td>92.1</td><td>93.2</td><td>·</td><td>96.4</td><td>89.0</td></tr><tr><td>AIME&#x27;24</td><td>85.7</td><td>81.4</td><td>74.3</td><td> 79.8</td><td>83.9</td><td>92.0</td><td>79.6</td></tr><tr><td>AIME&#x27;25</td><td>81.5</td><